In [2]:
# train_model.py
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

FEATURES = [
    "avg_kw", "peak_kw", "base_kw", "load_factor",
    "variability_cv", "peak_hour_mode",
    "peak_to_offpeak_ratio", "weekend_weekday_ratio",
    "hvac_share", "standby_share", "shiftable_share"
]

def build_building_features(meter_csv="data/meter_readings.csv"):
    m = pd.read_csv(meter_csv, parse_dates=["timestamp"])
    m["kwh"] = m["power_kw"] * 0.25

    bt = m.groupby(["building_id", "timestamp"], as_index=False)["power_kw"].sum()
    bt["hour"] = bt["timestamp"].dt.hour
    bt["dow"] = bt["timestamp"].dt.dayofweek
    bt["is_weekend"] = (bt["dow"] >= 5).astype(int)

    agg = bt.groupby("building_id")["power_kw"].agg(["mean", "max", "std"]).rename(
        columns={"mean": "avg_kw", "max": "peak_kw", "std": "std_kw"}
    )
    base = bt.groupby("building_id")["power_kw"].quantile(0.05).rename("base_kw")
    feat = agg.join(base)

    feat["load_factor"] = feat["avg_kw"] / feat["peak_kw"]
    feat["variability_cv"] = feat["std_kw"] / (feat["avg_kw"] + 1e-9)

    bt["is_peak_interval"] = bt.groupby("building_id")["power_kw"].transform(lambda s: s == s.max())
    peak_hours = bt[bt["is_peak_interval"]].groupby("building_id")["hour"].agg(
        lambda s: int(s.mode().iloc[0]) if len(s.mode()) else int(s.iloc[0])
    ).rename("peak_hour_mode")
    feat = feat.join(peak_hours)

    peak_mask = bt["hour"].between(18, 22)
    off_mask = bt["hour"].between(0, 6)
    peak_avg = bt[peak_mask].groupby("building_id")["power_kw"].mean().rename("peak_avg_kw")
    off_avg = bt[off_mask].groupby("building_id")["power_kw"].mean().rename("off_avg_kw")
    feat = feat.join(peak_avg).join(off_avg)
    feat["peak_to_offpeak_ratio"] = feat["peak_avg_kw"] / (feat["off_avg_kw"] + 1e-9)
    feat.drop(columns=["peak_avg_kw", "off_avg_kw"], inplace=True)

    wknd = bt[bt["is_weekend"] == 1].groupby("building_id")["power_kw"].mean().rename("wknd_kw")
    wkdy = bt[bt["is_weekend"] == 0].groupby("building_id")["power_kw"].mean().rename("wkdy_kw")
    feat = feat.join(wknd).join(wkdy)
    feat["weekend_weekday_ratio"] = feat["wknd_kw"] / (feat["wkdy_kw"] + 1e-9)
    feat.drop(columns=["wknd_kw", "wkdy_kw"], inplace=True)

    # Appliance shares
    app = m.groupby(["building_id", "appliance"], as_index=False)["kwh"].sum()
    total = app.groupby("building_id")["kwh"].sum().rename("total_kwh")
    app = app.merge(total, on="building_id", how="left")
    app["share"] = app["kwh"] / (app["total_kwh"] + 1e-9)

    hvac_share = app[app["appliance"] == "HVAC"].set_index("building_id")["share"].rename("hvac_share")
    standby_like = app[app["appliance"].isin(["Fridge", "TV", "Computers"])].groupby("building_id")["share"].sum().rename("standby_share")
    shiftable = app[app["appliance"].isin(["Washer", "Dryer", "Dishwasher", "Oven", "EVCharger", "WaterHeater", "Lighting"])].groupby("building_id")["share"].sum().rename("shiftable_share")

    feat = feat.join(hvac_share).join(standby_like).join(shiftable)
    feat[["hvac_share", "standby_share", "shiftable_share"]] = feat[["hvac_share", "standby_share", "shiftable_share"]].fillna(0.0)

    return feat.reset_index()

def sweep_kmeans(X, k_min=2, k_max=10, random_state=42):
    rows = []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        rows.append({"k": k, "inertia": float(km.inertia_), "silhouette": float(sil)})
    return pd.DataFrame(rows).sort_values("k")

def main():
    os.makedirs("artifacts", exist_ok=True)

    feat = build_building_features()
    X = feat[FEATURES].copy()

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # PCA: keep enough components to explain 90% variance
    pca = PCA(n_components=0.90, random_state=42)
    Xp = pca.fit_transform(Xs)

    # Save PCA explained variance (for UI)
    evr = pca.explained_variance_ratio_
    pca_ev = pd.DataFrame({
        "component": np.arange(1, len(evr) + 1),
        "explained_variance_ratio": evr,
        "cumulative_explained_variance": np.cumsum(evr)
    })
    pca_ev.to_csv("artifacts/pca_explained_variance.csv", index=False)

    # Save PCA loadings (interpretation)
    loadings = pd.DataFrame(
        pca.components_,
        columns=FEATURES,
        index=[f"PC{i+1}" for i in range(pca.n_components_)]
    )
    loadings.to_csv("artifacts/pca_loadings.csv")

    # K sweep (elbow + silhouette curves)
    k_sweep = sweep_kmeans(Xp, 2, 10, random_state=42)
    k_sweep.to_csv("artifacts/k_sweep.csv", index=False)

    # choose k with best silhouette
    best_row = k_sweep.sort_values("silhouette", ascending=False).iloc[0]
    best_k = int(best_row["k"])

    km = KMeans(n_clusters=best_k, n_init="auto", random_state=42)
    labels = km.fit_predict(Xp)

    # 2D PCA coordinates for scatter plots
    # (use first 2 PCs from the trained PCA; if only 1 PC, pad with zeros)
    pc1 = Xp[:, 0]
    pc2 = Xp[:, 1] if Xp.shape[1] > 1 else np.zeros_like(pc1)

    scores2d = pd.DataFrame({
        "building_id": feat["building_id"].values,
        "pc1": pc1,
        "pc2": pc2,
        "cluster": labels
    })
    scores2d.to_csv("artifacts/pca_scores_2d.csv", index=False)

    # Silhouette per building (for UI boxplots)
    sil_samples = silhouette_samples(Xp, labels)
    sil_df = pd.DataFrame({
        "building_id": feat["building_id"].values,
        "cluster": labels,
        "silhouette": sil_samples
    })
    sil_df.to_csv("artifacts/silhouette_samples.csv", index=False)

    # Cluster profiles (means of original features)
    segmented = feat.copy()
    segmented["cluster"] = labels
    segmented.to_csv("artifacts/building_segments.csv", index=False)

    prof = segmented.groupby("cluster")[FEATURES].mean()
    prof["count"] = segmented.groupby("cluster")["building_id"].size()
    prof = prof.reset_index()
    prof.to_csv("artifacts/cluster_profiles.csv", index=False)

    joblib.dump(scaler, "artifacts/scaler.joblib")
    joblib.dump(pca, "artifacts/pca.joblib")
    joblib.dump(km, "artifacts/kmeans.joblib")

    print(f"Saved artifacts. best_k={best_k}")
    print("Wrote: pca_explained_variance.csv, pca_loadings.csv, pca_scores_2d.csv, k_sweep.csv, silhouette_samples.csv, cluster_profiles.csv")

if __name__ == "__main__":
    main()


Saved artifacts. best_k=8
Wrote: pca_explained_variance.csv, pca_loadings.csv, pca_scores_2d.csv, k_sweep.csv, silhouette_samples.csv, cluster_profiles.csv
